![image](https://raw.githubusercontent.com/IBM/watsonx-ai-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use AutoAI RAG and Chroma to create a pattern and get information from `ibm-watsonx-ai` SDK documentation

#### Disclaimers

- Use only Projects and Spaces that are available in the watsonx context.


## Notebook content

This notebook contains the steps and code to demonstrate the usage of IBM AutoAI RAG. The AutoAI RAG experiment conducted in this notebook uses data scraped from the `ibm-watsonx-ai` SDK documentation.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goal

The learning goals of this notebook are:

- Create an AutoAI RAG job that will find the best RAG pattern based on provided data


## Table of Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [RAG Optimizer definition](#RAG-Optimizer-definition)
3. [Run the RAG Experiment](#Run-the-RAG-Experiment)
4. [Compare and test RAG Patterns](#Compare-and-test-RAG-Patterns)
5. [Historical runs](#Historical-runs)
6. [Cleanup](#Cleanup)
7. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup task:

-  Contact your Cloud Pak for Data administrator and ask them for your account credentials

### Install dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install -U "ibm-watsonx-ai[rag]" | tail -n 1

#### Define credentials

Authenticate the watsonx.ai Runtime service on IBM Cloud Pak for Data. You need to provide the **admin's** `username` and the platform `url`.

In [2]:
import os

try:
    username = os.environ["USERNAME"]
except KeyError:
    username = input("Please enter your username (hit enter): ")

try:
    url = os.environ["URL"]
except KeyError:
    url = input("Please enter the platform url (hit enter): ")

Use the **admin's** `api_key` to authenticate watsonx.ai Runtime services:

In [3]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai API key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.4",
)

Alternatively you can use the **admin's** `password`:

In [4]:
import getpass

from ibm_watsonx_ai import Credentials

if "credentials" not in locals() or not credentials.api_key:
    credentials = Credentials(
        username=username,
        password=getpass.getpass("Enter your watsonx.ai password and hit enter: "),
        url=url,
        instance_id="openshift",
        version="5.4",
    )

#### Create `APIClient` instance

In [5]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials)

### Working with spaces

First, you need to create a space for your work. If you do not have a space already created, you can use `{PLATFORM_URL}/ml-runtime/spaces?context=icp4data` to create one.

- Click **New Deployment Space**
- Create an empty space
- Go to the space `Settings` tab
- Copy `Space GUID` into your env file or else enter it in the window which will show up after running below cell

**Tip**: You can also use SDK to prepare the space for your work. Find more information in the [Space Management sample notebook](https://github.com/IBM/watson-machine-learning-samples/blob/master/cpd5.0/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: Assign the space ID below

In [6]:
try:
    space_id = os.environ["SPACE_ID"]
except KeyError:
    space_id = input("Please enter your space_id (hit enter): ")

To print all existing spaces, use the `list` method.

In [7]:
client.spaces.list(limit=10)

,ID,NAME,CREATED
0,76f10ef2-947e-4e83-8a83-d07e24f1cab1,ML deployment space,2026-09-02T08:39:55.967Z


To be able to interact with all resources available in watsonx.ai, you need to set the **space** which you will be using.

In [8]:
client.set.default_space(space_id)

'SUCCESS'

<a id="RAG-Optimizer-definition"></a>
## RAG Optimizer definition

### Define a connection to the training data

Upload the training data to the project as a data asset and then define a connection to the file. This example uses the `ModelInference` description from the [`ibm_watsonx_ai`](https://ibm.github.io/watsonx-ai-python-sdk/fm_model_inference.html) documentation.

In [9]:
from langchain_community.document_loaders import WebBaseLoader

url_file = "https://ibm.github.io/watsonx-ai-python-sdk/v1.3.42/fm_model_inference.html"

docs = WebBaseLoader(url_file).load()
model_inference_content = docs[0].page_content

Upload the training data to the project as a data asset.

In [10]:
document_filename = "ModelInference.txt"

if not os.path.isfile(document_filename):
    with open(document_filename, "w") as file:
        file.write(model_inference_content)

document_asset_details = client.data_assets.create(
    name=document_filename, file_path=document_filename
)

document_asset_id = client.data_assets.get_id(document_asset_details)
document_asset_id

Creating data asset...
SUCCESS


'01a0661b-d11c-71ab-af16-9ea53cc891c6'

Define a connection to the training data.

In [11]:
from ibm_watsonx_ai.helpers import DataConnection

input_data_references = [DataConnection(data_asset_id=document_asset_id)]

### Define a connection to the test data

Upload a `json` file that you want to use as a benchmark to the project as a data asset and then define a connection to the file. This example uses content from the [`ibm_watsonx_ai`](https://ibm.github.io/watsonx-ai-python-sdk/index.html) SDK documentation.

In [12]:
benchmarking_data_IBM_page_content = [
    {
        "question": "What is path to ModelInference class?",
        "correct_answer": "ibm_watsonx_ai.foundation_models.ModelInference",
        "correct_answer_document_ids": ["ModelInference.txt"],
    },
    {
        "question": "What is method for get model inference details?",
        "correct_answer": "get_details()",
        "correct_answer_document_ids": ["ModelInference.txt"],
    },
]

Upload the benchmark testing data to the project as a data asset with `json` extension.

In [13]:
import json

test_filename = "benchmarking_data_ModelInference.json"

if not os.path.isfile(test_filename):
    with open(test_filename, "w") as json_file:
        json.dump(benchmarking_data_IBM_page_content, json_file, indent=4)

test_asset_details = client.data_assets.create(
    name=test_filename, file_path=test_filename
)

test_asset_id = client.data_assets.get_id(test_asset_details)
test_asset_id

Creating data asset...
SUCCESS


'01a0661b-d81d-727c-b280-11f2de0a29f5'

Define a connection to the benchmark testing data.

In [14]:
test_data_references = [DataConnection(data_asset_id=test_asset_id)]

### Configure the RAG Optimizer

Provide the input information for the AutoAI RAG optimizer:
- `name` - experiment name
- `description` - experiment description
- `max_number_of_rag_patterns` - maximum number of RAG patterns to create
- `optimization_metrics` - target optimization metrics

In [15]:
from ibm_watsonx_ai.experiment import AutoAI
from ibm_watsonx_ai.foundation_models.schema import AutoAIRAGRetrievalConfig

experiment = AutoAI(
    credentials=credentials,
    space_id=space_id,
)

retrieval_config = AutoAIRAGRetrievalConfig(
    method="window",
    number_of_chunks=1,
    window_size=1,
)

chunking_config = {"method": "recursive", "chunk_size": 128, "chunk_overlap": 64}

rag_optimizer = experiment.rag_optimizer(
    name="AutoAI RAG test - sample noteook",
    description="Experiment run in sample notebook",
    chunking=[chunking_config],
    retrieval=[retrieval_config],
    max_number_of_rag_patterns=5,
    optimization_metrics=[AutoAI.RAGMetrics.ANSWER_CORRECTNESS],
)

To retrieve the configuration parameters, use `get_params()`.

In [16]:
rag_optimizer.get_params()

{'name': 'AutoAI RAG test - sample noteook',
 'description': 'Experiment run in sample notebook',
 'chunking': [{'method': 'recursive', 'chunk_size': 128, 'chunk_overlap': 64}],
 'max_number_of_rag_patterns': 5,
 'optimization_metrics': ['answer_correctness'],
 'retrieval': [{'method': 'window', 'number_of_chunks': 1, 'window_size': 1}]}

<a id="Run-the-RAG-Experiment"></a>
## Run the RAG Experiment

Call the `run()` method to trigger the AutoAI RAG experiment. Choose one of two modes: 

- To use the **interactive mode** (synchronous job), specify `background_mode=False` 
- To use the **background mode** (asynchronous job), specify `background_mode=True`

In [17]:
run_details = rag_optimizer.run(
    input_data_references=input_data_references,
    test_data_references=test_data_references,
    background_mode=False,
)



##############################################

Running '1044501a-5d68-4a32-9f23-3000d48471cd'

##############################################


pending...........
running.................
completed
Training of '1044501a-5d68-4a32-9f23-3000d48471cd' finished successfully.


To monitor the AutoAI RAG jobs in background mode, use the `get_run_status()` method.

In [18]:
rag_optimizer.get_run_status()

'completed'

<a id="Compare-and-test-RAG-Patterns"></a>
## Compare and test RAG Patterns

You can list the trained patterns and information on evaluation metrics in the form of a Pandas DataFrame by calling the `summary()` method. Use the DataFrame to compare all discovered patterns and select the one you want for further testing.

In [19]:
summary = rag_optimizer.summary()
summary

,mean_answer_correctness,chunking.method,chunking.chunk_size,chunking.chunk_overlap,embeddings.model_id,vector_store.distance_metric,retrieval.method,retrieval.number_of_chunks,generation.model_id,agent.type
Pattern_Name,,,,,,,,,,
Pattern1,0.5,recursive,128,64,ibm/slate-125m-english-rtrvr,cosine,window,1,ibm/granite-4-h-small,sequential


Additionally, you can pass the `scoring` parameter to the summary method to filter RAG patterns, starting with the best.

In [20]:
summary = rag_optimizer.summary(scoring="faithfulness")

### Get the selected pattern

Get the RAGPattern object from the RAG Optimizer experiment. By default, the RAGPattern of the best pattern is returned.

In [21]:
best_pattern_name = summary.index.values[0]
print("Best pattern is:", best_pattern_name)

best_pattern = rag_optimizer.get_pattern(pattern_name="Pattern1")

Best pattern is: Pattern1


To retrieve the pattern details, use the `get_pattern_details` method.

In [22]:
rag_optimizer.get_pattern_details(pattern_name="Pattern1")

{'composition_steps': ['model_selection',
  'chunking',
  'embedding',
  'retrieval',
  'generation',
  'optimization'],
 'duration_seconds': 105,
 'location': {'evaluation_results': '/spaces/76f10ef2-947e-4e83-8a83-d07e24f1cab1/assets/auto_ml/auto_ml.c7cdfa1b-c8c1-4e5c-9c65-d017c62dd60c/wml_data/1044501a-5d68-4a32-9f23-3000d48471cd/Pattern1/evaluation_results.json',
  'indexing_notebook': '/spaces/76f10ef2-947e-4e83-8a83-d07e24f1cab1/assets/auto_ml/auto_ml.c7cdfa1b-c8c1-4e5c-9c65-d017c62dd60c/wml_data/1044501a-5d68-4a32-9f23-3000d48471cd/Pattern1/indexing_inference_notebook.ipynb',
  'inference_notebook': '/spaces/76f10ef2-947e-4e83-8a83-d07e24f1cab1/assets/auto_ml/auto_ml.c7cdfa1b-c8c1-4e5c-9c65-d017c62dd60c/wml_data/1044501a-5d68-4a32-9f23-3000d48471cd/Pattern1/indexing_inference_notebook.ipynb',
  'inference_service_code': '/spaces/76f10ef2-947e-4e83-8a83-d07e24f1cab1/assets/auto_ml/auto_ml.c7cdfa1b-c8c1-4e5c-9c65-d017c62dd60c/wml_data/1044501a-5d68-4a32-9f23-3000d48471cd/Pattern1/

Query the RAGPattern locally to test it.

In [23]:
from ibm_watsonx_ai.deployments import RuntimeContext

runtime_context = RuntimeContext(api_client=client)

url = client.credentials.url
inference_service_function = best_pattern.inference_service(runtime_context, url=url)[0]

In [24]:
question = "How to add Task Credentials?"

context = RuntimeContext(
    api_client=client,
    request_payload_json={"messages": [{"role": "user", "content": question}]},
)

inference_service_function(context)

{'body': {'choices': [{'index': 0,
    'message': {'role': 'system',
     'content': 'To add task credentials for a deployed model using the ModelInference class in Python, you need to provide the deployment ID and the necessary credentials. Here\'s how you can do it:\n\n1. First, make sure you have the required dependencies installed. You\'ll need the ibm-watson-machine-learning library. You can install it using pip:\n\n   ```\n   pip install ibm-watson-machine-learning\n   ```\n\n2. Import the necessary classes from the ibm-watson-machine-learning library:\n\n   ```python\n   from ibm_watson_machine_learning import APIClient, ModelInference, Credentials\n   ```\n\n3. Create an instance of the APIClient class and set your IBM Cloud API key:\n\n   ```python\n   api_key = "<YOUR_IBM_CLOUD_API_KEY>"\n   client = APIClient(api_key=api_key)\n   ```\n\n   Replace `<YOUR_IBM_CLOUD_API_KEY>` with your actual IBM Cloud API key.\n\n4. Specify the deployment ID of the model you want to use:\n\n 

### Deploy the RAGPattern

To deploy the RAGPattern, store the defined RAG function and then create a deployed asset.

In [25]:
deployment_details = best_pattern.inference_service.deploy(
    name="AutoAI RAG deployment - ibm_watsonx_ai documentataion",
    space_id=space_id,
    deploy_params={"tags": ["wx-autoai-rag"]},
)



######################################################################################

Synchronous deployment creation for id: '01a06622-99db-731c-8a27-960261bb3217' started

######################################################################################


initializing
Note: online_url is deprecated and will be removed in a future release. Use serving_urls instead.
.....................
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='01a06622-c040-71da-8331-54f71f13ab5d'
-----------------------------------------------------------------------------------------------




### Test the deployed function

The RAG service is now deployed in our space. To test the solution, run the cell below. Questions have to be provided in the payload. Their format is provided below.

In [26]:
deployment_id = client.deployments.get_id(deployment_details)

question = "How to add Task Credentials?"

payload = {"messages": [{"role": "user", "content": question}]}
score_response = client.deployments.run_ai_service(deployment_id, payload)

In [27]:
print(score_response["choices"][0]["message"]["content"])

To add task credentials for a deployed model using the ModelInference class in Python, you need to provide the deployment ID and the necessary credentials. Here's how you can do it:

1. First, make sure you have the required libraries installed. You'll need the ibm-watson-machine-learning library. You can install it using pip:

   ```
   pip install ibm-watson-machine-learning
   ```

2. Import the necessary classes from the ibm-watson-machine-learning library:

   ```python
   from ibm_watson_machine_learning import APIClient, ModelInference, Credentials
   ```

3. Create an instance of the APIClient class and set your IBM Cloud API key:

   ```python
   api_key = "YOUR_IBM_CLOUD_API_KEY"
   client = APIClient(api_key=api_key)
   ```

   Replace `"YOUR_IBM_CLOUD_API_KEY"` with your actual IBM Cloud API key.

4. Specify the deployment ID of the model you want to use:

   ```python
   deployment_id = "<ID of deployed model>"
   ```

   Replace `"<ID of deployed model>"` with the actual 

<a id="Historical-runs"></a>
## Historical runs

In this section, you will learn how to work with historical RAG Optimizer jobs (runs).

To list historical runs, use the `list()` method and provide the `'rag_optimizer'` filter.

In [28]:
experiment.runs(filter="rag_optimizer").list()

,timestamp,run_id,state,auto_pipeline_optimizer name
0,2026-09-03T07:14:51.135Z,1044501a-5d68-4a32-9f23-3000d48471cd,completed,AutoAI RAG test - sample noteook


In [29]:
run_id = run_details["metadata"]["id"]
run_id

'1044501a-5d68-4a32-9f23-3000d48471cd'

### Get the executed optimizer's configuration parameters

In [30]:
experiment.runs.get_rag_params(run_id=run_id)

{'name': 'AutoAI RAG test - sample noteook',
 'description': 'Experiment run in sample notebook',
 'chunking': [{'chunk_overlap': 64, 'chunk_size': 128, 'method': 'recursive'}],
 'max_number_of_rag_patterns': 5,
 'retrieval': [{'method': 'window', 'number_of_chunks': 1, 'window_size': 1}],
 'optimization_metrics': ['answer_correctness']}

### Get the historical `rag_optimizer` instance and training details

In [31]:
historical_opt = experiment.runs.get_rag_optimizer(run_id)

### List trained patterns for the selected optimizer

In [32]:
historical_opt.summary()

,mean_answer_correctness,chunking.method,chunking.chunk_size,chunking.chunk_overlap,embeddings.model_id,vector_store.distance_metric,retrieval.method,retrieval.number_of_chunks,generation.model_id,agent.type
Pattern_Name,,,,,,,,,,
Pattern1,0.5,recursive,128,64,ibm/slate-125m-english-rtrvr,cosine,window,1,ibm/granite-4-h-small,sequential


<a id="Cleanup"></a>
## Cleanup

To delete the current experiment, use the `cancel_run(hard_delete=True)` method.

**Warning:** Be careful: once you delete an experiment, you will no longer be able to refer to it.

In [33]:
rag_optimizer.cancel_run(hard_delete=True)

'SUCCESS'

To delete the deployment, use the `delete` method. 

**Warning:** If you keep the deployment active, it might lead to unnecessary consumption of Compute Unit Hours (CUHs).

In [34]:
client.deployments.delete(deployment_id)

'SUCCESS'

To clean up all of the created assets:
- experiments
- trainings
- pipelines
- model definitions
- models
- functions
- deployments

follow the steps in this sample [notebook](https://github.com/IBM/watsonx-ai-samples/blob/master/cpd5.1/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to use `ibm-watsonx-ai` to run AutoAI RAG experiments. 

 Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors and Maintainers

**Mateusz Szewczyk (Former)**, Software Engineer at IBM watsonx.ai

**Paweł Kocur**, Software Engineer at IBM watsonx.ai

**Rafał Chrzanowski**, Software Engineer at IBM watsonx.ai

Copyright © 2025-2026 IBM. This notebook and its source code are released under the terms of the MIT License.